# Prachatai — Frozen WangchanBERTa + Optuna + 5 Runs (Run เพิ่ม 4 รอบ)

ไฟล์นี้ยึดจาก notebook ที่รันจริง  
`wangchanberta-x-transformers-prachatai-optimized-2(2).ipynb`

## Protocol เดิมที่คงไว้
- Dataset: official `prachatai_train.csv`, `prachatai_validation.csv`, `prachatai_test.csv`
- Text column: `body_text`
- 12 labels เดิม
- Encoder: `airesearch/wangchanberta-base-att-spm-uncased`
- **Frozen WangchanBERTa**
- Precompute/cache encoder hidden states เพียงครั้งเดียว
- X-Transformers decoder + Attention Pooling + classifier
- BCEWithLogitsLoss + `pos_weight = negative / positive`
- Batch size = 64
- Max length = 128
- Final training = 100 epochs
- Threshold tuning จาก validation set
- Optuna search space เดิม

## การทำ 5 runs แบบประหยัดเวลา
ผลเดิมใน notebook ต้นฉบับเป็น **Run 1, seed=42** และมี Optuna best params แล้ว  
ไฟล์นี้จึงใช้ผลเดิมเป็น Run 1 และ train ใหม่เพิ่มอีก 4 seeds:

`[13, 21, 87, 123]`

โดยใช้:
- official train/validation/test ชุดเดิม
- architecture เดิม
- Optuna best params เดิม
- threshold tuning ใหม่จาก validation ในแต่ละ run

ท้าย notebook จะคำนวณ:
- Mean
- Median
- Sample Variance (`ddof=1`)
- Sample Standard Deviation (`ddof=1`)
- per-label Precision / Recall / F1

> `RUN_OPTUNA_AGAIN=False` เป็นค่าเริ่มต้น เพราะ Optuna เดิมรันเสร็จแล้ว  
> ถ้าต้องการค้นหา best params ใหม่ค่อยเปลี่ยนเป็น `True`

In [ ]:
!pip install -q "transformers==4.44.2" "tokenizers<0.20" sentencepiece protobuf x-transformers optuna scikit-learn

In [ ]:
import os
import glob
import gc
import json
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss
)

import optuna

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

In [ ]:
# =========================================================
# CONFIG — คง setup เดิมจาก notebook ที่รันจริง
# =========================================================

BASE_MODEL_ID = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128
EPOCHS = 100
BATCH_SIZE = 64

NUM_LABELS = 12
N_TRIALS = 10

SEED = 42

LABEL_COLUMNS = [
    "politics",
    "human_rights",
    "quality_of_life",
    "international",
    "social",
    "environment",
    "economics",
    "culture",
    "labor",
    "national_security",
    "ict",
    "education"
]

TEXT_COLUMN = "body_text"

# =========================================================
# 5-RUN CONTROL
# =========================================================

# ค่า default: ใช้ผล seed=42 เดิมเป็น Run 1 แล้วรันเพิ่ม 4 รอบ
USE_EXISTING_RUN1 = True

ADDITIONAL_SEEDS = [13, 21, 87, 123]

# ถ้าต้องการรันใหม่ครบ 5 seeds ทั้งหมดภายหลัง
FULL_SEEDS = [13, 21, 42, 87, 123]

# Optuna เดิมทำเสร็จแล้ว จึงไม่ต้องรันใหม่
RUN_OPTUNA_AGAIN = False

# best params ที่ได้จาก Optuna ใน notebook ต้นฉบับ
ORIGINAL_BEST_PARAMS = {
    "depth": 2,
    "heads": 4,
    "attn_dropout": 0.2691450563663464,
    "ff_dropout": 0.28475714690185233,
    "ff_mult": 4,
    "lr": 0.0001955577124628116,
    "weight_decay": 0.0015755090667191358
}

OUTPUT_DIR = (
    "/kaggle/working"
    if os.path.exists("/kaggle/working")
    else "/content"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [ ]:
# =========================================================
# REPRODUCIBILITY
# =========================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

In [ ]:
# =========================================================
# FIND OFFICIAL PRACHATAI SPLITS
# =========================================================

def find_csv(filename):

    candidates = [
        f"/content/{filename}",
        f"/mnt/data/{filename}"
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    hits = glob.glob(
        f"/kaggle/input/**/{filename}",
        recursive=True
    )

    if hits:
        return hits[0]

    raise FileNotFoundError(
        f"ไม่พบ {filename} — "
        f"กรุณา Add dataset ใน Kaggle หรือ upload เข้า /content"
    )


TRAIN_CSV_PATH = find_csv(
    "prachatai_train.csv"
)

VALIDATION_CSV_PATH = find_csv(
    "prachatai_validation.csv"
)

TEST_CSV_PATH = find_csv(
    "prachatai_test.csv"
)

print("TRAIN CSV     :", TRAIN_CSV_PATH)
print("VALIDATION CSV:", VALIDATION_CSV_PATH)
print("TEST CSV      :", TEST_CSV_PATH)

In [ ]:
# =========================================================
# LOAD OFFICIAL TRAIN / VALIDATION / TEST
# =========================================================

required_columns = [
    TEXT_COLUMN
] + LABEL_COLUMNS


def load_prachatai_split(
    path,
    split_name
):

    frame = pd.read_csv(
        path,
        usecols=required_columns,
        low_memory=False
    )

    for col in required_columns:
        if col not in frame.columns:
            raise ValueError(
                f"{split_name} ไม่มี column: {col}"
            )

    frame = frame[
        required_columns
    ].copy()

    frame[TEXT_COLUMN] = (
        frame[TEXT_COLUMN]
        .astype(str)
        .str.strip()
    )

    for col in LABEL_COLUMNS:

        frame[col] = pd.to_numeric(
            frame[col],
            errors="coerce"
        ).fillna(0).astype(int)

        invalid = ~frame[col].isin(
            [0, 1]
        )

        if invalid.any():
            values = (
                frame.loc[
                    invalid,
                    col
                ]
                .unique()
                .tolist()
            )

            raise ValueError(
                f"{split_name}/{col} "
                f"มีค่านอกเหนือจาก 0/1: {values}"
            )

    return frame.reset_index(
        drop=True
    )


train_df = load_prachatai_split(
    TRAIN_CSV_PATH,
    "train"
)

val_df = load_prachatai_split(
    VALIDATION_CSV_PATH,
    "validation"
)

test_df = load_prachatai_split(
    TEST_CSV_PATH,
    "test"
)

print("TRAIN     :", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST      :", len(test_df))

In [ ]:
# =========================================================
# LOAD WANGCHANBERTA
# =========================================================

from huggingface_hub import snapshot_download

MODEL_NAME = None

for attempt in range(10):

    try:

        MODEL_NAME = snapshot_download(
            BASE_MODEL_ID,
            ignore_patterns=[
                "*.safetensors"
            ]
        )

        print(
            "MODEL PATH:",
            MODEL_NAME
        )

        break

    except Exception as exc:

        print(
            f"retry {attempt + 1}: {exc}"
        )

if MODEL_NAME is None:
    raise RuntimeError(
        "โหลด WangchanBERTa ไม่สำเร็จ"
    )


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [ ]:
# =========================================================
# FROZEN ENCODER + PRECOMPUTE EMBEDDINGS ONCE
# =========================================================

encoder = AutoModel.from_pretrained(
    MODEL_NAME
).to(device).eval()

for param in encoder.parameters():
    param.requires_grad = False

HIDDEN_SIZE = encoder.config.hidden_size

print("HIDDEN SIZE:", HIDDEN_SIZE)
print(
    "TRAINABLE ENCODER PARAMETERS:",
    sum(
        p.numel()
        for p in encoder.parameters()
        if p.requires_grad
    )
)


class RawDataset(Dataset):

    def __init__(self, frame):

        self.texts = (
            frame[TEXT_COLUMN]
            .tolist()
        )

        self.labels = (
            frame[LABEL_COLUMNS]
            .values
            .astype(np.float32)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(
        self,
        idx
    ):

        enc = tokenizer(
            str(
                self.texts[idx]
            ),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids":
                enc["input_ids"]
                .squeeze(0),

            "attention_mask":
                enc["attention_mask"]
                .squeeze(0),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.float
                )
        }


@torch.no_grad()
def precompute_embeddings(
    frame,
    name
):

    loader = DataLoader(
        RawDataset(frame),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    embeddings = []
    masks = []
    targets = []

    use_amp = (
        device.type == "cuda"
    )

    for batch in tqdm(
        loader,
        desc=f"Encoding {name}"
    ):

        input_ids = (
            batch["input_ids"]
            .to(device)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):

            output = encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        embeddings.append(
            output
            .last_hidden_state
            .half()
            .cpu()
        )

        masks.append(
            attention_mask.cpu()
        )

        targets.append(
            batch["labels"]
        )

    return (
        torch.cat(embeddings),
        torch.cat(masks),
        torch.cat(targets)
    )


train_emb, train_mask, train_labels = (
    precompute_embeddings(
        train_df,
        "train"
    )
)

val_emb, val_mask, val_labels = (
    precompute_embeddings(
        val_df,
        "validation"
    )
)

test_emb, test_mask, test_labels = (
    precompute_embeddings(
        test_df,
        "test"
    )
)

del encoder
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "TRAIN EMBEDDING:",
    tuple(train_emb.shape),
    train_emb.dtype
)

In [ ]:
# =========================================================
# CACHED DATASET + SEEDED DATALOADERS
# =========================================================

class CachedDataset(Dataset):

    def __init__(
        self,
        emb,
        mask,
        labels
    ):

        self.emb = emb
        self.mask = mask
        self.labels = labels

    def __len__(self):
        return len(self.emb)

    def __getitem__(
        self,
        idx
    ):

        return {
            "embeddings":
                self.emb[idx],

            "attention_mask":
                self.mask[idx],

            "labels":
                self.labels[idx]
        }


train_dataset = CachedDataset(
    train_emb,
    train_mask,
    train_labels
)

val_dataset = CachedDataset(
    val_emb,
    val_mask,
    val_labels
)

test_dataset = CachedDataset(
    test_emb,
    test_mask,
    test_labels
)


def make_loaders(seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )

In [ ]:
# =========================================================
# POS WEIGHT — สูตรเดิม negative / positive
# =========================================================

label_counts = (
    train_df[LABEL_COLUMNS]
    .sum()
    .values
    .astype(np.float32)
)

total_samples = len(
    train_df
)

neg_counts = (
    total_samples
    - label_counts
)

pos_weights = (
    neg_counts
    / (label_counts + 1e-6)
)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float
).to(device)

print(
    "LABEL COUNTS:",
    label_counts
)

print(
    "POS WEIGHTS:",
    pos_weights
)

In [ ]:
# =========================================================
# MODEL — เหมือน notebook ต้นฉบับ
# =========================================================

class AttentionPooling(nn.Module):

    def __init__(
        self,
        hidden_size
    ):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(
                hidden_size,
                hidden_size
            ),
            nn.Tanh(),
            nn.Linear(
                hidden_size,
                1
            )
        )

    def forward(
        self,
        x,
        mask
    ):

        scores = (
            self.attention(x)
            .squeeze(-1)
        )

        scores = scores.masked_fill(
            mask == 0,
            torch.finfo(
                scores.dtype
            ).min
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x
            * weights.unsqueeze(-1),
            dim=1
        )

        return pooled


class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        self.decoder = Decoder(
            dim=HIDDEN_SIZE,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(
            HIDDEN_SIZE
        )

        self.dropout = nn.Dropout(
            0.4
        )

        self.fc = nn.Linear(
            HIDDEN_SIZE,
            NUM_LABELS
        )

    def forward(
        self,
        embeddings,
        attention_mask
    ):

        # คง behavior จาก notebook เดิม
        x = self.decoder(
            embeddings.float()
        )

        x = self.pooling(
            x,
            attention_mask
        )

        x = self.dropout(x)

        return self.fc(x)

In [ ]:
# =========================================================
# EVALUATION
# =========================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    thresholds=None
):

    model.eval()

    all_labels = []
    all_probs = []

    for batch in loader:

        embeddings = (
            batch["embeddings"]
            .to(device)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )

        labels = (
            batch["labels"]
            .to(device)
        )

        logits = model(
            embeddings,
            attention_mask
        )

        probs = torch.sigmoid(
            logits
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_labels.append(
            labels.cpu().numpy()
        )

    all_probs = np.vstack(
        all_probs
    )

    all_labels = np.vstack(
        all_labels
    )

    if thresholds is None:
        thresholds = np.array(
            [0.5] * NUM_LABELS
        )

    preds = (
        all_probs
        >= thresholds
    ).astype(int)

    per_precision = precision_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_recall = recall_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_f1 = f1_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    return {
        "micro_f1":
            f1_score(
                all_labels,
                preds,
                average="micro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                all_labels,
                preds,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                all_labels,
                preds,
                average="weighted",
                zero_division=0
            ),

        "micro_precision":
            precision_score(
                all_labels,
                preds,
                average="micro",
                zero_division=0
            ),

        "macro_precision":
            precision_score(
                all_labels,
                preds,
                average="macro",
                zero_division=0
            ),

        "micro_recall":
            recall_score(
                all_labels,
                preds,
                average="micro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                all_labels,
                preds,
                average="macro",
                zero_division=0
            ),

        "subset_accuracy":
            accuracy_score(
                all_labels,
                preds
            ),

        "hamming_loss":
            hamming_loss(
                all_labels,
                preds
            ),

        "per_label_precision":
            per_precision,

        "per_label_recall":
            per_recall,

        "per_label_f1":
            per_f1,

        "labels":
            all_labels,

        "preds":
            preds,

        "probs":
            all_probs
    }

In [ ]:
# =========================================================
# TRAINING — 100 epochs ตาม notebook เดิม
# =========================================================

def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay,
    epochs=EPOCHS,
    verbose=True
):

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weights
    )

    trainable_params = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = (
        len(train_loader)
        * epochs
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(
                0.1 * total_steps
            ),
            num_training_steps=total_steps
        )
    )

    use_amp = (
        device.type == "cuda"
    )

    scaler = (
        torch.cuda.amp.GradScaler(
            enabled=use_amp
        )
    )

    for epoch in range(
        epochs
    ):

        model.train()

        total_loss = 0.0

        for batch in train_loader:

            embeddings = (
                batch["embeddings"]
                .to(device)
            )

            attention_mask = (
                batch["attention_mask"]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                logits = model(
                    embeddings,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(
                loss
            ).backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            scheduler.step()

            total_loss += (
                loss.item()
            )

        if verbose and (
            epoch == 0
            or (epoch + 1) % 10 == 0
            or epoch == epochs - 1
        ):

            val_result = evaluate(
                model,
                val_loader
            )

            print(
                f"Epoch {epoch+1:3d}/{epochs} | "
                f"loss={total_loss:.4f} | "
                f"val_micro={val_result['micro_f1']:.4f} | "
                f"val_macro={val_result['macro_f1']:.4f} | "
                f"val_acc={val_result['subset_accuracy']:.4f}"
            )

    return model

In [ ]:
# =========================================================
# THRESHOLD TUNING — เหมือน notebook เดิม
# 0.10 ถึง <0.90 step 0.05
# =========================================================

def optimize_thresholds(
    y_true,
    y_probs
):

    best_thresholds = []

    for i in range(
        NUM_LABELS
    ):

        best_thr = 0.5
        best_f1 = 0.0

        for thr in np.arange(
            0.1,
            0.9,
            0.05
        ):

            pred = (
                y_probs[:, i]
                >= thr
            ).astype(int)

            score = f1_score(
                y_true[:, i],
                pred,
                zero_division=0
            )

            if score > best_f1:

                best_f1 = score
                best_thr = thr

        best_thresholds.append(
            best_thr
        )

    return np.array(
        best_thresholds
    )

In [ ]:
# =========================================================
# OPTIONAL OPTUNA
# =========================================================
# ค่า default ไม่รันซ้ำ เพราะ best params จาก notebook เดิมมีอยู่แล้ว

best_params = (
    ORIGINAL_BEST_PARAMS.copy()
)

if RUN_OPTUNA_AGAIN:

    print(
        "RUNNING OPTUNA AGAIN..."
    )

    optuna_train_loader, (
        optuna_val_loader
    ), _ = make_loaders(
        SEED
    )

    def objective(trial):

        # reset randomness ก่อนแต่ละ trial
        # เพื่อให้ trial comparison reproducible ขึ้น
        seed_everything(SEED)

        depth = trial.suggest_int(
            "depth",
            1,
            2
        )

        heads = trial.suggest_categorical(
            "heads",
            [2, 4]
        )

        attn_dropout = trial.suggest_float(
            "attn_dropout",
            0.2,
            0.5
        )

        ff_dropout = trial.suggest_float(
            "ff_dropout",
            0.2,
            0.5
        )

        ff_mult = trial.suggest_int(
            "ff_mult",
            2,
            4
        )

        lr = trial.suggest_float(
            "lr",
            1e-5,
            2e-4,
            log=True
        )

        weight_decay = trial.suggest_float(
            "weight_decay",
            1e-4,
            1e-2,
            log=True
        )

        model = EmotionModel(
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        ).to(device)

        model = train_model(
            model,
            optuna_train_loader,
            optuna_val_loader,
            lr,
            weight_decay,
            epochs=EPOCHS,
            verbose=False
        )

        val_result = evaluate(
            model,
            optuna_val_loader
        )

        thresholds = optimize_thresholds(
            val_result["labels"],
            val_result["probs"]
        )

        tuned = evaluate(
            model,
            optuna_val_loader,
            thresholds
        )

        score = tuned[
            "macro_f1"
        ]

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return score


    sampler = (
        optuna.samplers.TPESampler(
            seed=SEED
        )
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS
    )

    best_params = (
        study.best_params
    )

    print(
        "NEW BEST PARAMS:",
        best_params
    )

else:

    print(
        "Using original Optuna best params:"
    )

    print(
        best_params
    )

In [ ]:
# =========================================================
# IMPORT ORIGINAL RUN 1 (seed=42)
# =========================================================
# ค่าจาก output ใน notebook ต้นฉบับ
# ใช้เป็น Run 1 เพื่อไม่ต้อง train seed 42 ซ้ำ

EXISTING_RUN1 = {
    "run": 1,
    "seed": 42,
    "subset_accuracy": 0.3517,
    "micro_precision": 0.6023,
    "micro_recall": 0.7563,
    "micro_f1": 0.6706,
    "macro_precision": 0.5471,
    "macro_recall": 0.7100,
    "macro_f1": 0.6165,
    "weighted_f1": 0.6762,
    "hamming_loss": 0.1006
}

# classification_report เดิม
EXISTING_RUN1_PER_LABEL = pd.DataFrame({
    "label": LABEL_COLUMNS,

    "precision": [
        0.74,
        0.59,
        0.56,
        0.63,
        0.33,
        0.65,
        0.51,
        0.42,
        0.73,
        0.41,
        0.56,
        0.42
    ],

    "recall": [
        0.86,
        0.68,
        0.75,
        0.84,
        0.51,
        0.74,
        0.66,
        0.61,
        0.88,
        0.63,
        0.78,
        0.57
    ],

    "f1": [
        0.80,
        0.63,
        0.64,
        0.72,
        0.41,
        0.69,
        0.57,
        0.50,
        0.80,
        0.49,
        0.65,
        0.49
    ],

    "support": [
        3842,
        1511,
        1127,
        834,
        789,
        772,
        519,
        398,
        350,
        338,
        292,
        255
    ]
})

print("ORIGINAL RUN 1")
print(EXISTING_RUN1)

display(
    EXISTING_RUN1_PER_LABEL
)

In [ ]:
# =========================================================
# RUN ADDITIONAL 4 SEEDS AUTOMATICALLY
# =========================================================

overall_rows = []
per_label_rows = []

if USE_EXISTING_RUN1:

    overall_rows.append(
        EXISTING_RUN1.copy()
    )

    tmp = (
        EXISTING_RUN1_PER_LABEL
        .copy()
    )

    tmp.insert(
        0,
        "seed",
        42
    )

    tmp.insert(
        0,
        "run",
        1
    )

    per_label_rows.append(
        tmp
    )

    seeds_to_run = (
        ADDITIONAL_SEEDS
    )

    start_run = 2

else:

    seeds_to_run = (
        FULL_SEEDS
    )

    start_run = 1


for offset, seed in enumerate(
    seeds_to_run
):

    run_number = (
        start_run + offset
    )

    print("\n" + "=" * 90)
    print(
        f"RUN {run_number}/5 | seed={seed}"
    )
    print("=" * 90)

    seed_everything(
        seed
    )

    (
        train_loader,
        val_loader,
        test_loader
    ) = make_loaders(
        seed
    )

    model = EmotionModel(
        depth=best_params["depth"],
        heads=best_params["heads"],
        attn_dropout=best_params["attn_dropout"],
        ff_dropout=best_params["ff_dropout"],
        ff_mult=best_params["ff_mult"]
    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        epochs=EPOCHS,
        verbose=True
    )

    # tune threshold บน official validation ของ run นี้
    val_result = evaluate(
        model,
        val_loader
    )

    thresholds = optimize_thresholds(
        val_result["labels"],
        val_result["probs"]
    )

    test_result = evaluate(
        model,
        test_loader,
        thresholds
    )

    row = {
        "run":
            run_number,

        "seed":
            seed,

        "subset_accuracy":
            test_result[
                "subset_accuracy"
            ],

        "micro_precision":
            test_result[
                "micro_precision"
            ],

        "micro_recall":
            test_result[
                "micro_recall"
            ],

        "micro_f1":
            test_result[
                "micro_f1"
            ],

        "macro_precision":
            test_result[
                "macro_precision"
            ],

        "macro_recall":
            test_result[
                "macro_recall"
            ],

        "macro_f1":
            test_result[
                "macro_f1"
            ],

        "weighted_f1":
            test_result[
                "weighted_f1"
            ],

        "hamming_loss":
            test_result[
                "hamming_loss"
            ]
    }

    overall_rows.append(
        row
    )

    print("\nTEST RESULT")

    for key, value in row.items():

        if key not in [
            "run",
            "seed"
        ]:

            print(
                f"{key:20s}: "
                f"{value:.6f}"
            )

    print("\nTHRESHOLDS")

    for label, thr in zip(
        LABEL_COLUMNS,
        thresholds
    ):

        print(
            f"{label:20s}: "
            f"{thr:.2f}"
        )

    for i, label in enumerate(
        LABEL_COLUMNS
    ):

        per_label_rows.append(
            pd.DataFrame({
                "run": [
                    run_number
                ],

                "seed": [
                    seed
                ],

                "label": [
                    label
                ],

                "precision": [
                    test_result[
                        "per_label_precision"
                    ][i]
                ],

                "recall": [
                    test_result[
                        "per_label_recall"
                    ][i]
                ],

                "f1": [
                    test_result[
                        "per_label_f1"
                    ][i]
                ],

                "support": [
                    int(
                        test_result[
                            "labels"
                        ][:, i]
                        .sum()
                    )
                ]
            })
        )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


results_df = pd.DataFrame(
    overall_rows
)

per_label_df = pd.concat(
    per_label_rows,
    ignore_index=True
)

print("\nALL 5 RUNS")
display(
    results_df
)

In [ ]:
# =========================================================
# MEAN / MEDIAN / VARIANCE / STANDARD DEVIATION
# =========================================================

metric_columns = [
    "subset_accuracy",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "hamming_loss"
]

summary_rows = []

for metric in metric_columns:

    values = (
        results_df[metric]
        .astype(float)
    )

    summary_rows.append({
        "metric":
            metric,

        "mean":
            values.mean(),

        "median":
            values.median(),

        # sample variance
        "variance":
            values.var(
                ddof=1
            ),

        # sample standard deviation
        "std":
            values.std(
                ddof=1
            ),

        "min":
            values.min(),

        "max":
            values.max()
    })


summary_df = pd.DataFrame(
    summary_rows
)

print(
    "\n5-RUN STATISTICAL SUMMARY"
)

display(
    summary_df
)

In [ ]:
# =========================================================
# PER-LABEL SUMMARY ACROSS 5 RUNS
# =========================================================

per_label_summary = (
    per_label_df
    .groupby(
        "label",
        sort=False
    )
    .agg(
        precision_mean=(
            "precision",
            "mean"
        ),

        precision_variance=(
            "precision",
            lambda x: x.var(
                ddof=1
            )
        ),

        precision_std=(
            "precision",
            lambda x: x.std(
                ddof=1
            )
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_variance=(
            "recall",
            lambda x: x.var(
                ddof=1
            )
        ),

        recall_std=(
            "recall",
            lambda x: x.std(
                ddof=1
            )
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_median=(
            "f1",
            "median"
        ),

        f1_variance=(
            "f1",
            lambda x: x.var(
                ddof=1
            )
        ),

        f1_std=(
            "f1",
            lambda x: x.std(
                ddof=1
            )
        )
    )
    .reset_index()
)

display(
    per_label_summary
)

In [ ]:
# =========================================================
# EXPORT RESULTS
# =========================================================

raw_path = os.path.join(
    OUTPUT_DIR,
    "prachatai_5runs_raw_results.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "prachatai_5runs_summary.csv"
)

per_label_raw_path = os.path.join(
    OUTPUT_DIR,
    "prachatai_5runs_per_label_raw.csv"
)

per_label_summary_path = os.path.join(
    OUTPUT_DIR,
    "prachatai_5runs_per_label_summary.csv"
)

best_params_path = os.path.join(
    OUTPUT_DIR,
    "prachatai_optuna_best_params_used.json"
)

results_df.to_csv(
    raw_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

per_label_df.to_csv(
    per_label_raw_path,
    index=False,
    encoding="utf-8-sig"
)

per_label_summary.to_csv(
    per_label_summary_path,
    index=False,
    encoding="utf-8-sig"
)

with open(
    best_params_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        best_params,
        file,
        ensure_ascii=False,
        indent=2
    )


print("EXPORTED")
print(raw_path)
print(summary_path)
print(per_label_raw_path)
print(per_label_summary_path)
print(best_params_path)